# 00. Data Protocol

Purpose: fix the data universe, split policy, leakage rules, and
output layout before any tokenizer, scaler, model, or vocabulary is
fit.

Stable protocol logic lives in `kairos.experiments.protocol` and artifact/hash
helpers live in `kairos.experiments.artifacts`.


## Research Gates

- D1 uses one stock index as one independent dataset/run.
- D2 uses n stock indexes from one country or market group as one
  dataset/run.
- D3 uses all selected global major stock indexes as one combined
  dataset/run.
- Domestic Korean index data uses Kiwoom from `1990-01-01` onward.
- Overseas index data uses KIS and keeps all rows returned by the
  endpoint.
- No scaler, tokenizer, model, vocabulary, ATR normalization, z-score,
  or volume median is fit before time-based boundaries are fixed.


In [37]:
from brokers.kis import download_overseas_index_info

indices = download_overseas_index_info()

indices[indices["korean_name"].str.contains("나스닥|S&P500|다우존스", na=False)]

,class_code,symbol,english_name,korean_name,industry_code,is_dow30,is_nasdaq100,is_sp500,exchange_code,country_code,raw_source,downloaded_at
45,P,.DJI,DOW JONES INDUSTRIAL AVERAGE INDEX,다우존스 산업지수,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
46,P,.DJINET,DOW JONES COMPOSITE INTERNET INDEX,다우존스 인터넷,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
47,P,.DJT,DOW JONES TRANSPORTATION INDEX,다우존스 운송지수,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
48,P,.DJU,DOW JONES UTILITIES INDEX,다우존스 유틸리티,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
49,P,BANK,NASDAQ Bank,나스닥 은행,,False,False,False,NASD,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
50,P,COMP,NASDAQ Composite,나스닥 종합,,False,False,False,NASD,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
51,P,DJC,BBG Commodity Index,다우존스 UBS 상품지수,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
53,P,DJR,DOW JONES EQUITY REIT INDEX,다우존스 부동산신탁,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
71,P,DJUSAP,DOW JONES US AUTOMOBILES INDEX,다우존스 자동차제조,,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00
72,P,DJUSAT,DOW JONES US AUTO PARTS INDEX,다우존스 자동차부품,AUP,False,False,False,DOWJ,501,frgn_code.mst,2026-07-06T12:52:58.805035+00:00


In [38]:
from __future__ import annotations

from datetime import datetime, timezone

from kairos.experiments.artifacts import config_hash, find_project_root, strip_hash_excluded, to_jsonable
from kairos.experiments.protocol import (
    DATASET_REGISTRY,
    DOMESTIC_KIWOOM_START_DATE,
    INDEX_SYMBOLS,
    KIS_EARLIEST_START_DATE,
    PHASE_00_ID,
    RESEARCH_NAME,
    SPLIT_PROTOCOL,
    STEP_00_ID,
    build_manifest,
    build_protocol_config,
    empty_data_quality_metrics,
    figure_directory,
    run_directory,
)

PROJECT_ROOT = find_project_root()
RUNS_ROOT = PROJECT_ROOT / "notebooks" / "runs" / RESEARCH_NAME
FIGURES_ROOT = PROJECT_ROOT / "notebooks" / "figures" / RESEARCH_NAME

PROJECT_ROOT


PosixPath('/Users/choeseoggyu/portfolio/kairos')

## Index Dataset Registry


In [39]:
dataset_ids = list[str](DATASET_REGISTRY)
assert len(dataset_ids) == len(set(dataset_ids)), "dataset ids must be unique"
assert all(len(dataset.symbols) == 1 for dataset in DATASET_REGISTRY.values() if dataset.stage == "D1")
assert all(len(dataset.symbols) > 1 for dataset in DATASET_REGISTRY.values() if dataset.stage in {"D2", "D3"})
assert INDEX_SYMBOLS["KOSPI"].source.provider == "kiwoom"
assert INDEX_SYMBOLS["KOSDAQ"].source.provider == "kiwoom"
assert all(symbol.source.provider == "kis" for symbol in INDEX_SYMBOLS.values() if symbol.country != "KR")
assert DATASET_REGISTRY["d1_dji_daily"].symbols[0].symbol == "DJI"

dataset_summary = [
    {
        "dataset_id": dataset.dataset_id,
        "stage": dataset.stage,
        "interval": dataset.interval,
        "symbols": [item.symbol for item in dataset.symbols],
        "providers": sorted(dataset.provider_set),
        "broker_mapping": dataset.broker_mapping,
    }
    for dataset in DATASET_REGISTRY.values()
]
dataset_summary


[{'dataset_id': 'd1_kospi_daily',
  'stage': 'D1',
  'interval': '1d',
  'symbols': ['KOSPI'],
  'providers': ['kiwoom'],
  'broker_mapping': {'KOSPI': {'provider': 'kiwoom',
    'period_method': 'client.domestic.chart.industry_daily',
    'minute_method': 'client.domestic.chart.industry_minute',
    'period_symbol': '001',
    'minute_symbol': '001',
    'master_symbol': None}}},
 {'dataset_id': 'd1_kospi_1m',
  'stage': 'D1',
  'interval': '1m',
  'symbols': ['KOSPI'],
  'providers': ['kiwoom'],
  'broker_mapping': {'KOSPI': {'provider': 'kiwoom',
    'period_method': 'client.domestic.chart.industry_daily',
    'minute_method': 'client.domestic.chart.industry_minute',
    'period_symbol': '001',
    'minute_symbol': '001',
    'master_symbol': None}}},
 {'dataset_id': 'd1_kosdaq_daily',
  'stage': 'D1',
  'interval': '1d',
  'symbols': ['KOSDAQ'],
  'providers': ['kiwoom'],
  'broker_mapping': {'KOSDAQ': {'provider': 'kiwoom',
    'period_method': 'client.domestic.chart.industry_dail

## Split and Leakage Protocol


In [40]:
assert SPLIT_PROTOCOL.train_end < SPLIT_PROTOCOL.validation_start
assert SPLIT_PROTOCOL.validation_start < SPLIT_PROTOCOL.validation_end
assert SPLIT_PROTOCOL.validation_end < SPLIT_PROTOCOL.test_start
assert SPLIT_PROTOCOL.embargo_days >= max(SPLIT_PROTOCOL.label_horizons)
assert DOMESTIC_KIWOOM_START_DATE == "1990-01-01"
assert KIS_EARLIEST_START_DATE == "1900-01-01"

SPLIT_PROTOCOL


SplitProtocol(train_start='2005-01-01', train_end='2016-12-31', validation_start='2017-01-01', validation_end='2020-12-31', test_start='2021-01-01', test_end=None, embargo_days=20, label_horizons=(1, 5, 20), rolling_statistics_rule='statistics for row t use only rows <= t-1; model fit uses train split only')

## Config Hash and Output Layout


In [41]:
example_dataset = DATASET_REGISTRY["d1_dji_daily"]
example_config = build_protocol_config(example_dataset)
example_hash = config_hash(example_config)
example_run_dir = run_directory(
    RUNS_ROOT,
    example_dataset,
    example_hash,
    seed=7,
    started_at="20260705-000000",
    phase_id=PHASE_00_ID,
    step_id=STEP_00_ID,
)
example_figure_dir = figure_directory(
    FIGURES_ROOT,
    example_dataset,
    example_hash,
    phase_id=PHASE_00_ID,
    step_id=STEP_00_ID,
)

assert "notebooks/runs/candlestick-shape-quantization" in example_run_dir.as_posix()
assert "phase-00-data-protocol" in example_run_dir.as_posix()
assert "cfg-" in example_run_dir.as_posix()
example_hash, example_run_dir, example_figure_dir


('716f04de',
 PosixPath('/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-00-data-protocol/step-01-index-universe-and-split/d1_dji_daily/cfg-716f04de/run-20260705-000000_seed-7'),
 PosixPath('/Users/choeseoggyu/portfolio/kairos/notebooks/figures/candlestick-shape-quantization/phase-00-data-protocol/step-01-index-universe-and-split/d1_dji_daily/cfg-716f04de/selected'))

## Manifest and Metrics Scaffolding


In [42]:
example_manifest = build_manifest(example_dataset, strip_hash_excluded(to_jsonable(example_config)), example_hash, seed=7)
example_manifest["created_at_utc"] = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
example_metrics = empty_data_quality_metrics(example_dataset)

assert example_manifest["cfg_hash"] == example_hash
assert example_metrics["leakage_checks"]["random_split_forbidden"] is True
example_manifest, example_metrics


({'research': 'candlestick-shape-quantization',
  'phase': 'phase-00-data-protocol',
  'step': 'step-01-index-universe-and-split',
  'dataset_id': 'd1_dji_daily',
  'dataset_stage': 'D1',
  'cfg_hash': '716f04de',
  'seed': 7,
  'source_notebook': 'notebooks/candlestick-shape-quantization/00_data_protocol.ipynb',
  'symbols': [{'symbol': 'DJI',
    'name': 'Dow Jones Industrial Average',
    'country': 'US',
    'source': {'provider': 'kis',
     'period_method': 'client.overseas.chart.major_index',
     'minute_method': 'client.overseas.chart.index_minute',
     'period_symbol': '.DJI',
     'minute_symbol': '.DJI',
     'master_symbol': '.DJI',
     'note': 'KIS README uses .DJI for major_index examples'}}],
  'split': {'train_start': '2005-01-01',
   'train_end': '2016-12-31',
   'validation_start': '2017-01-01',
   'validation_end': '2020-12-31',
   'test_start': '2021-01-01',
   'test_end': None,
   'embargo_days': 20,
   'label_horizons': (1, 5, 20),
   'rolling_statistics_rule':

## Broker Smoke Evidence

These are observed API availability facts. The actual data-fetching
code lives in `01_shape_feature_validation.ipynb`.


In [43]:
BROKER_SMOKE_EVIDENCE = {
    "d1_kospi_daily": {
        "provider": "kiwoom",
        "method": "client.domestic.chart.industry_daily",
        "broker_symbol": "001",
        "request": {"base_date": "2026-07-03", "start_date": "1990-01-01", "max_pages": None},
        "row_count": 9420,
        "date_range": {"start": "1990-01-03", "end": "2026-07-03"},
        "observed_openapi_behavior": "Kiwoom continued pagination until the 1990-01-01 project start-date boundary.",
        "feature_validation_cfg_hash": "c5854c22",
    },
    "d1_dji_daily": {
        "provider": "kis",
        "method": "client.overseas.chart.major_index",
        "broker_symbol": ".DJI",
        "request": {"start": "1900-01-01", "end": "2026-07-03", "period": "D"},
        "row_count": 100,
        "date_range": {"start": "2026-02-09", "end": "2026-07-02"},
        "observed_openapi_behavior": "KIS major-index endpoint returned a 100-row window even with an earlier requested start date.",
        "feature_validation_cfg_hash": "0ead053d",
    },
    "kis_candidate_notes": {
        "index_minute_rows": {"SPX": 102, "DJI": 0, "IXIC": 0},
    },
}

assert BROKER_SMOKE_EVIDENCE["d1_kospi_daily"]["row_count"] > 9000
assert BROKER_SMOKE_EVIDENCE["d1_dji_daily"]["row_count"] == 100
assert BROKER_SMOKE_EVIDENCE["d1_kospi_daily"]["date_range"]["start"] >= "1990-01-01"
BROKER_SMOKE_EVIDENCE


{'d1_kospi_daily': {'provider': 'kiwoom',
  'method': 'client.domestic.chart.industry_daily',
  'broker_symbol': '001',
  'request': {'base_date': '2026-07-03',
   'start_date': '1990-01-01',
   'max_pages': None},
  'row_count': 9420,
  'date_range': {'start': '1990-01-03', 'end': '2026-07-03'},
  'observed_openapi_behavior': 'Kiwoom continued pagination until the 1990-01-01 project start-date boundary.',
  'feature_validation_cfg_hash': 'c5854c22'},
 'd1_dji_daily': {'provider': 'kis',
  'method': 'client.overseas.chart.major_index',
  'broker_symbol': '.DJI',
  'request': {'start': '1900-01-01', 'end': '2026-07-03', 'period': 'D'},
  'row_count': 100,
  'date_range': {'start': '2026-02-09', 'end': '2026-07-02'},
  'observed_openapi_behavior': 'KIS major-index endpoint returned a 100-row window even with an earlier requested start date.',
  'feature_validation_cfg_hash': '0ead053d'},
 'kis_candidate_notes': {'index_minute_rows': {'SPX': 102,
   'DJI': 0,
   'IXIC': 0}}}

## Protocol Audit Checklist

- Confirm Kiwoom availability for domestic index codes and KIS
  availability for overseas index codes.
- Confirm every D1/D2/D3 dataset has enough train, validation, and
  test rows.
- Record data-quality metrics and split row counts per symbol.
- Freeze the split dates or revise this protocol before any fitting.
- Write `experiment_config.json`, `metrics.json`, and `manifest.json`
  for each data-availability run.
